# 🛡️ FinGuard v0.2.0-Elite: LLM Safety for Financial AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/protectai/finguard/blob/main/notebooks/FinGuard_Demo.ipynb)

Welcome to the **FinGuard Elite** demo! FinGuard is an enterprise-grade safety layer specifically tuned for fintech and wealth management.

### 🌟 v0.2.0-Elite Features:
- **Instant Fast-Path**: Sub-1ms regex detection for PAN, Aadhaar, IFSC, and VPAs.
- **PMLA Compliance**: Anti-Money Laundering logic to flag high-value (>₹50k) transfers.
- **Context-Aware Guardrails**: Distinguishes between banking operations (Pass) and investment advice (Requires Disclaimer).
- **Tiered Performance**: Sub-40ms AI safety on CPU using optimized ONNX models.

---

### 📦 1. Installation
First, we'll install FinGuard and its dependencies. This takes ~1 minute as it sets up the ONNX runtime environment.

In [ ]:
# In Colab, install directly from GitHub (or pip if published)
!pip install git+https://github.com/protectai/finguard
!pip install onnxruntime-extensions  # Required for some ONNX operations

### 📑 2. Global Safety Initialization
We'll initialize **FinGuard** with an elite policy configuration.

In [ ]:
from finguard import FinGuard
import asyncio

# Enterprise Elite Policy
policy_config = {
    "policy_id": "elite_banking_v2",
    "pii": {
        "enabled": True, 
        "fast_pii_only": False, 
        "redact_output": True # Redact response PII
    },
    "injection": {"enabled": True},
    "output": {"on_fail": "block"}
}

guard = FinGuard(policy=policy_config)

# Mock LLM to simulate financial responses
async def mock_llm(prompt: str) -> str:
    await asyncio.sleep(0.01)
    if "balance" in prompt:
        return "Your balance is ₹45,200."
    if "invest" in prompt:
        return "I suggest buying HDFC Stock for 20% returns!"
    return f"Response to: {prompt}"

### 🎭 3. Dynamic Persona Testing
FinGuard adapts its safety checks based on the use case. Let's test three distinct personas.

In [ ]:
async def test_persona(name, prompt):
    print(f"\n>>> Testing Persona: {name}")
    print(f"    Prompt: {prompt}")
    try:
        res = await guard(prompt, llm_fn=mock_llm)
        if res.is_safe:
            print(f"    ✅ Result: PASS | Latency: {res.latency_ms:.1f}ms")
            print(f"    LLM says: {res.output}")
        else:
            print(f"    ❌ Result: BLOCKED | Latency: {res.latency_ms:.1f}ms")
            print(f"    Reason: {res.violations[0]['scanner']}")
    except Exception as e:
        print(f"    ⚠️ Error: {e}")

# Scenario 1: Casual Banker (Retail Operations)
await test_persona("Casual Banker", "Show me my transaction history")

# Scenario 2: Compliance Violation (PMLA Check)
await test_persona("Compliance Officer", "Transfer ₹250,000 to account 12345")

# Scenario 3: Wealth Manager (Anonymization check)
await test_persona("Wealth Manager", "My PAN is ABCDE1234F, tell me about my returns.")

### 📊 4. Tiered Latency Breakdown
One of FinGuard's unique features is **Transparency**. We can see exactly which component of the safety audit took how long.

In [ ]:
res = await guard("Should I invest in crypto?", llm_fn=mock_llm)
print("Component Latencies (ms):")
for component, lat in res.component_latencies.items():
    print(f"{component:<30}: {lat:.2f}ms")